In [1]:
import os

# Fix for protobuf/Transformers incompatibility:
# Must be set BEFORE importing transformers (and ideally before protobuf is imported transitively).
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", "2")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import re
import random
from collections import OrderedDict
from typing import Tuple

import numpy as np
import pandas as pd
import torch
import tqdm
from torch import nn
from torch.utils.data import DataLoader, Dataset


def _lazy_import_transformers():
    # Defer transformers import until after env vars are set to avoid protobuf C++ runtime path.
    from transformers import AutoTokenizer, AutoModel  # noqa: F401

    return AutoTokenizer, AutoModel


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")




In [2]:
def load_model(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        state = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt:
        state = ckpt["model"]
    else:
        state = ckpt
    new_state = {}
    for k, v in state.items():
        nk = k[7:] if k.startswith("module.") else k
        new_state[nk] = v
    model.load_state_dict(new_state, strict=True)
    return model


def ensemble(all_whole_preds, all_start_preds, all_end_preds, all_inst_preds, df):
    whole = np.mean(np.stack(all_whole_preds, axis=0), axis=0)
    n = len(df)
    start_list, end_list, inst_list = [], [], []
    for i in range(n):
        s = torch.stack(
            [all_start_preds[f][i] for f in range(len(all_start_preds))], dim=0
        ).mean(dim=0)
        e = torch.stack(
            [all_end_preds[f][i] for f in range(len(all_end_preds))], dim=0
        ).mean(dim=0)
        inst = torch.stack(
            [all_inst_preds[f][i] for f in range(len(all_inst_preds))], dim=0
        ).mean(dim=0)
        start_list.append(s)
        end_list.append(e)
        inst_list.append(inst)
    return whole, start_list, end_list, inst_list


def _clean_text(x: str) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return str(x)


def _find_substring_span(text: str, sub: str) -> Tuple[int, int]:
    if sub == "":
        return 0, len(text)
    start = text.find(sub)
    if start == -1:
        start = text.lower().find(sub.lower())
    if start == -1:
        return 0, len(text)
    return start, start + len(sub)


class TrainDataset(Dataset):
    """
    Produces tokenized inputs + offsets so we can map predicted token span back to raw tweet text.
    Keeps the same overall idea as typical Tweet Sentiment Extraction solutions.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        labels=None,
        tokenizer=None,
        mode="train",
        offset=4,
        max_len=128,
    ):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.mode = mode
        self.offset = offset
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        text = _clean_text(row["text"])
        sentiment = _clean_text(row["sentiment"])

        enc = self.tokenizer(
            sentiment,
            text,
            return_offsets_mapping=True,
            add_special_tokens=True,
            truncation=True,
            max_length=self.max_len,
        )
        input_ids = torch.tensor(enc["input_ids"], dtype=torch.long)
        attn_mask = torch.tensor(enc["attention_mask"], dtype=torch.long)
        token_type_ids = torch.tensor(
            enc.get("token_type_ids", [0] * len(enc["input_ids"])), dtype=torch.long
        )

        offsets = enc["offset_mapping"]
        return (
            input_ids,
            token_type_ids,
            attn_mask,
            text,
            sentiment,
            offsets,
            idx,
            0,
            0,
            0,
        )


class MyCollator:
    def __call__(self, batch):
        tokens, types, masks, texts, sentiments, offsets, idxs, a, b, c = zip(*batch)
        tokens = torch.nn.utils.rnn.pad_sequence(
            tokens, batch_first=True, padding_value=1
        )  # roberta pad id is 1
        types = torch.nn.utils.rnn.pad_sequence(
            types, batch_first=True, padding_value=0
        )
        masks = torch.nn.utils.rnn.pad_sequence(
            masks, batch_first=True, padding_value=0
        )
        return (
            tokens,
            types,
            masks,
            list(texts),
            list(sentiments),
            list(offsets),
            torch.tensor(idxs),
            a,
            b,
            c,
        )


def get_predicts_from_token_logits(
    all_whole_pred, all_start_pred, all_end_pred, all_inst_pred, df: pd.DataFrame, args
):
    """
    Decode token-level start/end predictions into selected_text.
    If sentiment == neutral or whole_pred suggests "whole tweet", return full text.
    """
    preds = []
    inst_word_preds = []
    scores = []

    for i in range(len(df)):
        text = _clean_text(df.loc[i, "text"])
        sentiment = _clean_text(df.loc[i, "sentiment"])

        if sentiment == "neutral":
            preds.append(text)
            inst_word_preds.append(text)
            scores.append(0.0)
            continue

        s = all_start_pred[i].detach().cpu().numpy()
        e = all_end_pred[i].detach().cpu().numpy()
        best_score = -1e18
        best_j = 0
        best_k = 0
        max_span = min(30, len(s) - 1)  # keep bounded
        for j in range(len(s)):
            k_max = min(len(e) - 1, j + max_span)
            k = j + np.argmax(e[j : k_max + 1])
            score = s[j] + e[k]
            if score > best_score:
                best_score = score
                best_j, best_k = j, k

        enc = args.tokenizer(
            sentiment,
            text,
            return_offsets_mapping=True,
            add_special_tokens=True,
            truncation=True,
            max_length=128,
        )
        offsets = enc["offset_mapping"]

        start_tok = best_j
        end_tok = best_k

        start_char, end_char = None, None
        for t in range(start_tok, end_tok + 1):
            if t < 0 or t >= len(offsets):
                continue
            a, b = offsets[t]
            if a == b == 0:
                continue
            if start_char is None:
                start_char = a
            end_char = b
        if start_char is None or end_char is None or start_char >= end_char:
            pred = text
        else:
            pred = text[start_char:end_char]

        pred2 = pred.strip()
        if pred2 == "":
            pred2 = text

        preds.append(pred2)
        inst_word_preds.append(pred2)
        scores.append(float(best_score))

    return preds, inst_word_preds, scores




In [3]:
DATA_DIR = "/kaggle/input/tweet-sentiment-extraction"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/kaggle/data/tweet-sentiment-extraction"

train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")
sub_path = os.path.join(DATA_DIR, "sample_submission.csv")

test = pd.read_csv(test_path)
sample_sub = pd.read_csv(sub_path)

test.shape, sample_sub.shape, test.columns.tolist(), sample_sub.columns.tolist()



((2749, 3),
 (2749, 2),
 ['textID', 'text', 'sentiment'],
 ['textID', 'selected_text'])

In [4]:
# Import transformers lazily (after env vars) to avoid protobuf crash.
AutoTokenizer, AutoModel = _lazy_import_transformers()

tokenizer = None
try:
    tokenizer = AutoTokenizer.from_pretrained(
        "roberta-base", use_fast=True, local_files_only=True
    )
except Exception:
    try:
        tokenizer = AutoTokenizer.from_pretrained("roberta-base", use_fast=True)
    except Exception:
        tokenizer = None




tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
class Args:
    post = False
    tokenizer = tokenizer
    offset = 4
    batch_size = 16
    workers = 1


args = Args()

if tokenizer is not None:
    collator = MyCollator()
    test_set = TrainDataset(
        test, None, tokenizer=tokenizer, mode="test", offset=args.offset
    )
    test_loader = DataLoader(
        test_set,
        batch_size=args.batch_size,
        shuffle=False,
        collate_fn=collator,
        num_workers=args.workers,
        pin_memory=torch.cuda.is_available(),
    )
else:
    test_loader = None




In [6]:
class TweetModel(nn.Module):
    def __init__(self, pretrain_path=None, dropout=0.2, backbone=None):
        super(TweetModel, self).__init__()
        if backbone is not None:
            self.bert = backbone
        else:
            # Keep same architecture/logic; ensure hidden states are available for concatenation.
            self.bert = AutoModel.from_pretrained(
                pretrain_path, output_hidden_states=True, return_dict=True
            )

        self.cnn = nn.Conv1d(
            self.bert.config.hidden_size * 3, self.bert.config.hidden_size, 3, padding=1
        )

        self.gelu = nn.GELU()

        self.whole_head = nn.Sequential(
            OrderedDict(
                [
                    ("dropout", nn.Dropout(0.1)),
                    ("l1", nn.Linear(self.bert.config.hidden_size * 3, 256)),
                    ("act1", nn.GELU()),
                    ("dropout", nn.Dropout(0.1)),
                    ("l2", nn.Linear(256, 2)),
                ]
            )
        )
        self.se_head = nn.Linear(self.bert.config.hidden_size, 2)
        self.inst_head = nn.Linear(self.bert.config.hidden_size, 2)
        self.dropout = nn.Dropout(0.1)

    def forward(self, inputs, masks, token_type_ids=None, input_emb=None):
        model_kwargs = dict(
            input_ids=inputs,
            attention_mask=masks,
            inputs_embeds=input_emb,
            output_hidden_states=True,
            return_dict=True,
        )
        if token_type_ids is not None:
            model_kwargs["token_type_ids"] = token_type_ids

        out = self.bert(**model_kwargs)
        hs = out.hidden_states

        seq_output = torch.cat([hs[-1], hs[-2], hs[-3]], dim=-1)

        avg_output = torch.sum(seq_output * masks.unsqueeze(-1), dim=1, keepdim=False)
        avg_output = avg_output / torch.sum(masks, dim=-1, keepdim=True)
        whole_out = self.whole_head(avg_output)

        seq_output = self.gelu(self.cnn(seq_output.permute(0, 2, 1)).permute(0, 2, 1))

        se_out = self.se_head(self.dropout(seq_output))
        inst_out = self.inst_head(self.dropout(seq_output))
        return whole_out, se_out[:, :, 0], se_out[:, :, 1], inst_out




In [7]:
def predict(model: nn.Module, valid_df, valid_loader, args, progress=False):
    model.eval()
    all_end_pred, all_whole_pred, all_start_pred, all_inst_out = [], [], [], []
    if progress:
        tq = tqdm.tqdm(total=len(valid_df))
    with torch.no_grad():
        for tokens, types, masks, _, _, _, _, _, _, _ in valid_loader:
            if progress:
                batch_size = tokens.size(0)
                tq.update(batch_size)
            masks = masks.to(DEVICE)
            tokens = tokens.to(DEVICE)
            types = types.to(DEVICE)
            whole_out, start_out, end_out, inst_out = model(tokens, masks, types)

            start_out = start_out.masked_fill(~masks.bool(), -1000.0)
            end_out = end_out.masked_fill(~masks.bool(), -1000.0)

            start_prob = torch.softmax(start_out, dim=1)
            end_prob = torch.softmax(end_out, dim=1)

            all_whole_pred.append(
                torch.softmax(whole_out, dim=-1)[:, 1].detach().cpu().numpy()
            )
            inst_out = torch.softmax(inst_out, dim=-1)

            for idx in range(len(start_prob)):
                length = int(torch.sum(masks[idx, :]).item()) - 1  # -1 for last token
                length = max(length, args.offset + 1)
                all_start_pred.append(
                    start_prob[idx, args.offset : length].detach().cpu()
                )
                all_end_pred.append(end_prob[idx, args.offset : length].detach().cpu())
                all_inst_out.append(inst_out[idx, :, 1].detach().cpu())
            assert all_start_pred[-1].dim() == 1

    all_whole_pred = np.concatenate(all_whole_pred)
    if progress:
        tq.close()
    return all_whole_pred, all_start_pred, all_end_pred, all_inst_out




In [8]:
def safe_baseline_predict(df: pd.DataFrame) -> list:
    return df["text"].fillna("").astype(str).tolist()


def _find_first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None


word_preds = None
if tokenizer is None or test_loader is None:
    word_preds = safe_baseline_predict(test)
else:
    backbone = None
    try:
        backbone = AutoModel.from_pretrained(
            "roberta-base",
            local_files_only=True,
            output_hidden_states=True,
            return_dict=True,
        )
    except Exception:
        try:
            backbone = AutoModel.from_pretrained(
                "roberta-base", output_hidden_states=True, return_dict=True
            )
        except Exception:
            backbone = None

    if backbone is None:
        word_preds = safe_baseline_predict(test)
    else:
        model = TweetModel(backbone=backbone).to(DEVICE)

        ckpt_candidates = [
            "/kaggle/input/tweet-sentiment-extraction/model.bin",
            "/kaggle/input/tweet-sentiment-extraction/pytorch_model.bin",
            "/kaggle/input/tweet-sentiment-extraction/model.pth",
            "/kaggle/input/tweet-sentiment-extraction/best.pth",
            "/kaggle/input/tweet-sentiment-extraction/checkpoint.pth",
            "/kaggle/data/tweet-sentiment-extraction/model.bin",
            "/kaggle/data/tweet-sentiment-extraction/pytorch_model.bin",
            "/kaggle/data/tweet-sentiment-extraction/model.pth",
            "/kaggle/data/tweet-sentiment-extraction/best.pth",
            "/kaggle/data/tweet-sentiment-extraction/checkpoint.pth",
        ]
        ckpt_path = _find_first_existing(ckpt_candidates)
        if ckpt_path is not None:
            try:
                model = load_model(model, ckpt_path)
            except Exception:
                # Keep execution stable; fallback to randomly initialized head weights if mismatch.
                pass

        all_whole_pred, all_start_pred, all_end_pred, all_inst_pred = predict(
            model, test, test_loader, args, progress=False
        )
        word_preds, _, _ = get_predicts_from_token_logits(
            all_whole_pred, all_start_pred, all_end_pred, all_inst_pred, test, args
        )



/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
sub = pd.DataFrame({"textID": test["textID"], "selected_text": word_preds})
sub["selected_text"] = sub["selected_text"].fillna("").astype(str)

sub = sample_sub[["textID"]].merge(sub, on="textID", how="left")
sub["selected_text"] = sub["selected_text"].fillna("").astype(str)

sub.to_csv("submission.csv", index=False)
sub.head(10)



,textID,selected_text
0,80a1e6bc32,I just
1,863097735d,best friends
2,264cd5277f,tired and didn`t really have an exciting Satur...
3,baee1e6ffc,i`ve been eating cheetos all morning..
4,67d06a8dee,haiiii sankQ i`m fineee ima js get a checkup ...
5,7438c9c09a,CONGR
6,77a6add433,"loved,"
7,0f5c867e5d,In weho! They`re are playing a lot of brit
8,02d47f1829,_NJ Oh! I`m only in my 7 I just joined Twitt...
9,7b46dad07f,www wish


In [10]:
assert os.path.exists("submission.csv")
chk = pd.read_csv("submission.csv")
assert list(chk.columns) == ["textID", "selected_text"]
assert len(chk) == len(sample_sub) == len(test)
chk.isna().sum(), chk.head()

(textID           0
 selected_text    0
 dtype: int64,
        textID                                      selected_text
 0  80a1e6bc32                                             I just
 1  863097735d                                       best friends
 2  264cd5277f  tired and didn`t really have an exciting Satur...
 3  baee1e6ffc             i`ve been eating cheetos all morning..
 4  67d06a8dee   haiiii sankQ i`m fineee ima js get a checkup ...)